# 6-LSH — L1: head hai scale

**Người thực hiện: PhuongTU**

B1 + WDice; decoder giữ nguyên. L0 dùng head cũ, L1 trung bình logits /1 + /2, L2 trung bình logits /1 + /2 + /4. Notebook chỉ clone source GitHub tại SHA đã khóa và gọi launcher `.py`. Chưa kết hợp SRR. Mỗi notebook chạy một session độc lập, mặc định 2 GPU, batch 4/GPU (effective batch 8), FP32, 100 epochs. Chỉ chạy cell train sau khi logger và preflight qua. Không có kết quả train dài được khẳng định ở đây.

## 1. Dependencies
Bật Internet và GPU. Giữ torch/torchvision/CUDA do Kaggle cấp. Nếu pip yêu cầu restart kernel, restart rồi chạy lại các cell theo thứ tự. Phải Add Input PhenoBench; notebook không tự tải dataset.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "pytorch-lightning==1.9.5", "torchmetrics==0.11.4", "timm==0.9.16",
    "setuptools<70", "numpy<2", "tensorboard", "matplotlib", "pandas", "tabulate",
    "scikit-image", "tifffile", "pyyaml", "rich"], check=True)

## 2. Clone source và chuẩn bị run
Giữ source SHA đã khóa. Dataset phải có train/val images + semantics và test/images. Chỉ điền đủ ba trường RESUME từ cùng bundle khi muốn tiếp tục run; mặc định fresh, không tự tìm last.ckpt.

In [ ]:
import pathlib
import re
import uuid

SCENARIO = "P3_LSH_L1_WDice"
GITHUB_REPO = "https://github.com/FUAVX/RPD_Reproduce.git"
GITHUB_BRANCH = "f/lsh"
SOURCE_COMMIT = "e39b25018f27bf107a406d90270245ee80b0b233"
CONFIG_RELATIVE = "configs/p3_scenarios/P3_LSH_L1_WDice.yaml"
DATASET_DIR = ""  # Empty means auto-detect exactly one attached PhenoBench root.
NUM_DEVICES = 2
NUM_WORKERS = 2
RESUME_CHECKPOINT = ""
RESUME_CONFIG = ""
RESUME_MANIFEST = ""

if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
    raise ValueError("Use the full reviewed source commit SHA")
RUN_TOKEN = uuid.uuid4().hex[:12]
RUN_DIR = pathlib.Path("/kaggle/working") / SCENARIO / RUN_TOKEN
RUN_DIR.mkdir(parents=True, exist_ok=False)
REPO_DIR = RUN_DIR / "source"
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--single-branch",
                "--branch", GITHUB_BRANCH, GITHUB_REPO, str(REPO_DIR)], check=True)
for arguments in (["sparse-checkout", "init", "--cone"],
                  ["sparse-checkout", "set", "models", "datasets", "callbacks", "configs", "tools", "utils"],
                  ["checkout", "--detach", SOURCE_COMMIT]):
    subprocess.run(["git", *arguments], cwd=REPO_DIR, check=True)
# Import launcher/model tools only from the clean pinned source.
import importlib.util
spec = importlib.util.spec_from_file_location("lsh_launcher_" + RUN_TOKEN,
                                            REPO_DIR / "tools/lsh_launcher.py")
launcher_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(launcher_module)
launch = launcher_module.LSHRun(REPO_DIR, RUN_DIR, SOURCE_COMMIT, CONFIG_RELATIVE,
    num_devices=NUM_DEVICES, num_workers=NUM_WORKERS, dataset_dir=DATASET_DIR,
    resume_checkpoint=RESUME_CHECKPOINT, resume_config=RESUME_CONFIG,
    resume_manifest=RESUME_MANIFEST)
assert launch.manifest["scenario"] == SCENARIO
print("Artifacts:", RUN_DIR)
print("Checkpoint rule:", launch.manifest["checkpoint_rule"])
print("Runtime hook SHA256:", launch.manifest["tensorboard_runtime_fix"]["startup_sha256"])

## 3. TensorBoard logger smoke test
Launcher sao startup hook notf ra ngoài Git checkout, truyền PYTHONPATH và P3_TENSORBOARD_NO_TF=1 cho subprocess/worker DDP. Test thật phải ghi/đọc scalar + grouped events và Lightning logger; không import TensorFlow/JAX thật.

In [ ]:
launch.logger_smoke()

## 4. Preflight riêng L1
Chạy các subprocess mới: core normal-import/chẵn-lẻ/backward/WDice; train sample thật lặp batch 4 với Adam, validation sample native lặp batch 4; DDP hai bước với find_unused_parameters=False. Không dùng preflight SRR hoặc model khác. Loader gốc không áp dụng center_crop validation trong YAML. Xem actual shapes và peak memory trước train; OOM phải dừng, không tự đổi recipe. Với NUM_DEVICES=1, smoke DDP dùng CPU và ghi rõ GPU-DDP chưa thử.

In [ ]:
launch.preflight()
print(launch.manifest["data_shapes"])
print(launch.manifest["preflight"]["gpu_preflight"])

## 5. Train
Chỉ chạy chủ động sau khi xem preflight. Giữ B1/WDice/recipe; best val_weed_iou (max) là checkpoint rule chung L0/L1/L2. Chưa tự thêm small-region coverage/miss rate. Cell này khởi động training dài; quá trình triển khai notebook không chạy cell này.

In [ ]:
launch.train()

## 6. Đóng gói output
Chạy lại sau ngắt có kiểm soát để lưu manifest, config, runtime hook, versions, logs, preflight và checkpoints. Source không được sao vào ZIP. Nếu runtime bị kill hoàn toàn, finally không được bảo đảm; giữ output đã lưu và dùng đúng bundle khi resume.

In [ ]:
artifact_path = launch.package()
print("Download:", artifact_path)